# 🎬 쇼츠 재료 만들기 (구글 코랩)

구글 드라이브에 올려 둔 **영화 원본(고화질·긴 파일 OK)**을 AI가 통째로 읽고,
**명화관식 쇼츠로 만들기 좋은 이야기 3개**를 추천해요. 번호만 고르면 필요한 장면들을 잘라 이어 붙여서
PC방 크롬에서 가볍게 열리는 **720p 재료 영상 + 자막 + 메모**를 드라이브 `쇼츠재료` 폴더에 저장해요.

### 사용법 (▶ 버튼 3번)
1. **①** ▶ → 드라이브 연결 허용 → 영화 목록이 나와요.
2. **②** 영화 번호를 넣고 ▶ → OpenAI API 키를 물어보면 넣기 → AI 추천 3개가 나와요.
3. **③** 마음에 드는 추천 번호를 넣고 ▶ → 몇 분 뒤 `쇼츠재료` 폴더에 저장돼요.

그다음 쇼츠 제작기에서 그 mp4·srt를 불러오고, 메모 칸에 저장된 메모를 붙여 넣으면 돼요.

In [ ]:
#@title ① 드라이브 연결하고 영화 찾기 { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
import os, re, json, subprocess, html, time, unicodedata, shutil

VIDEO_EXT = ('.mp4', '.mkv', '.avi', '.mov', '.m4v', '.ts', '.wmv', '.webm')
SUB_EXT = ('.smi', '.srt', '.ass', '.ssa', '.vtt')
OUT_DIR_NAME = '쇼츠재료'


def nfc(x): return unicodedata.normalize('NFC', x)


def find_files(root, max_files=400, min_gb=0.25):
    """드라이브 안의 영화(0.25GB 이상)·자막 파일을 찾아요 (쇼츠재료·숨김 폴더 제외)."""
    vids, subs = [], []
    skip = {nfc(x) for x in ('Colab Notebooks', OUT_DIR_NAME, 'Google AI Studio')}
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if not x.startswith('.') and nfc(x) not in skip]
        for f in files:
            p = os.path.join(d, f); low = f.lower()
            try:
                if low.endswith(VIDEO_EXT) and os.path.getsize(p) >= min_gb * 1073741824: vids.append(p)
                elif low.endswith(SUB_EXT): subs.append(p)
            except OSError:
                pass
        if len(vids) + len(subs) > max_files: break
    vids.sort(key=lambda p: os.path.basename(p).lower()); subs.sort(key=lambda p: os.path.basename(p).lower())
    return vids, subs


def hms(t):
    t = max(0, int(round(t))); return f'{t//3600}:{t%3600//60:02d}:{t%60:02d}'


def parse_time(s):
    s = str(s).strip().replace('：', ':')
    if not s: return None
    parts = [float(x) for x in s.split(':')]
    v = 0
    for x in parts: v = v * 60 + x
    return v


def read_text(path):
    b = open(path, 'rb').read()
    for enc in ('utf-8-sig', 'cp949', 'euc-kr', 'utf-16'):
        try:
            t = b.decode(enc)
            if enc.startswith('utf-8') and '�' in t: continue
            return t
        except Exception:
            pass
    return b.decode('utf-8', 'ignore')


def _clean(t):
    t = re.sub(r'(?i)<br\s*/?>', '\n', t)
    t = re.sub(r'<[^>]+>', '', t)
    t = html.unescape(t).replace('\xa0', ' ')
    t = '\n'.join(x.strip() for x in t.split('\n'))
    return re.sub(r'\n+', '\n', t).strip()


def parse_smi(txt):
    out = []
    items = re.findall(r'(?is)<sync\s+start\s*=\s*["\']?(\d+)[^>]*>(.*?)(?=<sync|</body|$)', txt)
    # 한국어 자막만 (여러 언어가 섞인 smi)
    has_kr = any(re.search(r'(?i)class\s*=\s*["\']?(krcc|kr)', b) for _, b in items)
    for i, (ms, body) in enumerate(items):
        if has_kr:
            m = re.search(r'(?is)<p[^>]*class\s*=\s*["\']?(krcc|kr)[^>]*>(.*)', body)
            body = m.group(2) if m else ''
        text = _clean(body)
        st = int(ms) / 1000
        en = int(items[i + 1][0]) / 1000 if i + 1 < len(items) else st + 3
        if text: out.append({'start': st, 'end': min(en, st + 10), 'text': text})
    return out


def parse_srt(txt):
    out = []
    for blk in re.split(r'\n\s*\n', txt.replace('\r', '')):
        m = re.search(r'(\d+):(\d+):(\d+)[,.](\d+)\s*-->\s*(\d+):(\d+):(\d+)[,.](\d+)', blk)
        if not m: continue
        g = [int(x) for x in m.groups()]
        st = g[0] * 3600 + g[1] * 60 + g[2] + g[3] / 1000
        en = g[4] * 3600 + g[5] * 60 + g[6] + g[7] / 1000
        text = _clean(blk[m.end():])
        if text: out.append({'start': st, 'end': en, 'text': text})
    return out


def load_subs(path):
    txt = read_text(path)
    if path.lower().endswith('.smi') or '<sync' in txt.lower(): return parse_smi(txt)
    if path.lower().endswith(('.ass', '.ssa', '.vtt')):
        tmp = '/tmp/_conv.srt'
        subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', path, tmp], check=False)
        if os.path.exists(tmp): return parse_srt(read_text(tmp))
        return []
    return parse_srt(txt)


def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration:stream=index,codec_type,codec_name,width,height:stream_tags=language',
                        '-of', 'json', path], capture_output=True, text=True)
    j = json.loads(r.stdout or '{}')
    dur = float(j.get('format', {}).get('duration', 0) or 0)
    v = next((s for s in j.get('streams', []) if s.get('codec_type') == 'video'), {})
    subs = [s for s in j.get('streams', []) if s.get('codec_type') == 'subtitle']
    return {'dur': dur, 'vcodec': v.get('codec_name', ''), 'w': v.get('width', 0), 'h': v.get('height', 0), 'subs': subs}


def embedded_kor_subs(path, info):
    for s in info['subs']:
        lang = (s.get('tags') or {}).get('language', '')
        if s.get('codec_name') in ('subrip', 'ass', 'ssa', 'mov_text', 'webvtt', 'text') and re.match(r'(?i)ko', lang or ''):
            tmp = '/tmp/_emb.srt'
            subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', path, '-map', f"0:{s['index']}", '-f', 'srt', tmp], check=False)
            if os.path.exists(tmp): return parse_srt(read_text(tmp))
    return []


def to_srt(subs):
    def ts(t):
        t = max(0, t); ms = int(round(t * 1000))
        return f'{ms//3600000:02d}:{ms%3600000//60000:02d}:{ms%60000//1000:02d},{ms%1000:03d}'
    return '\n'.join(f'{i+1}\n{ts(s["start"])} --> {ts(s["end"])}\n{s["text"]}\n' for i, s in enumerate(subs))


def cut_720p(src, start, end, out, log=print):
    """원하는 구간을 720p H.264 mp4로 잘라요 (PC방 크롬에서 가볍게 열리는 형식)."""
    dur = end - start
    cmd = ['ffmpeg', '-v', 'error', '-stats_period', '5', '-progress', 'pipe:1', '-y',
           '-ss', str(start), '-i', src, '-t', str(dur),
           '-map', '0:v:0', '-map', '0:a:0?', '-sn', '-dn',
           '-vf', "scale=-2:'min(720,ih)'", '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '22', '-pix_fmt', 'yuv420p',
           '-c:a', 'aac', '-b:a', '160k', '-ac', '2', '-movflags', '+faststart', out]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    last = 0
    for line in p.stdout:
        if line.startswith('out_time_ms='):
            try:
                t = int(line.split('=')[1]) / 1e6
                if time.time() - last > 5:
                    log(f'  자르는 중… {min(100, t/dur*100):.0f}%'); last = time.time()
            except ValueError:
                pass
    p.wait()
    if p.returncode != 0 or not os.path.exists(out):
        raise RuntimeError('자르기에 실패했어요: ' + p.stderr.read()[-400:])


def make_material(movie, sub_path, start, end, out_root, log=print):
    info = probe(movie)
    if not info['dur']: raise RuntimeError('영화 파일을 읽지 못했어요.')
    if start is None or end is None or end <= start: raise RuntimeError('시작·끝 시간을 확인해 주세요. 예) 1:36:30 ~ 1:41:30')
    end = min(end, info['dur'])
    log(f'영화: {os.path.basename(movie)} · 전체 {hms(info["dur"])} · {info["vcodec"].upper()} {info["w"]}x{info["h"]}')
    subs = load_subs(sub_path) if sub_path else embedded_kor_subs(movie, info)
    log(f'자막: {len(subs)}줄' + ('' if subs else ' (자막이 없어요. 웹 도구에서 받아쓰기를 켜면 돼요)'))
    os.makedirs(out_root, exist_ok=True)
    base = re.sub(r'[\\/:*?"<>|]', '', os.path.splitext(os.path.basename(movie))[0])[:40]
    name = f'{base}_{hms(start).replace(":", "")}-{hms(end).replace(":", "")}'
    out_mp4 = os.path.join(out_root, name + '.mp4'); out_srt = os.path.join(out_root, name + '.srt')
    t0 = time.time(); log(f'{hms(start)} ~ {hms(end)} ({(end-start)/60:.1f}분) 720p로 자르는 중…')
    tmp = '/tmp/' + name + '.mp4'
    cut_720p(movie, start, end, tmp, log)
    clip = [{'start': s['start'] - start, 'end': s['end'] - start, 'text': s['text']} for s in subs if s['end'] > start and s['start'] < end]
    for s in clip: s['start'] = max(0, s['start'])
    open('/tmp/' + name + '.srt', 'w', encoding='utf-8').write(to_srt(clip))
    import shutil
    shutil.copy('/tmp/' + name + '.mp4', out_mp4); shutil.copy('/tmp/' + name + '.srt', out_srt)
    mb = os.path.getsize(out_mp4) / 1048576
    log(f'완료! ({time.time()-t0:.0f}초) 영상 {mb:.0f}MB · 자막 {len(clip)}줄')
    log(f'구글 드라이브 > {OUT_DIR_NAME} 폴더에 저장했어요:\n  {name}.mp4\n  {name}.srt')
    return out_mp4, out_srt


def pretty_name(path):
    """파일 이름에서 화질·코덱 같은 꼬리표를 떼고 보기 좋게."""
    n = nfc(os.path.splitext(os.path.basename(path))[0])
    n = re.sub(r'[._]+', ' ', n)
    n = re.split(r'(?i)\b(19|20)\d\d\b|\b(1080p|720p|2160p|4k|bluray|web[- ]?dl|x264|x265|hevc|h264|nf|amzn|av1|10bit|hdr|dts|aac)\b', n)[0].strip(' -[]()')
    m = re.search(r'(?i)\b(19|20)\d\d\b', nfc(os.path.basename(path)))
    return (n or os.path.basename(path)) + (f' ({m.group(0)})' if m else '')


def _key(x): return re.sub(r'[^0-9a-z가-힣]', '', nfc(x).lower())


def match_sub(movie, subs):
    """영화와 이름이 비슷하거나 같은 폴더에 있는 자막을 찾아요."""
    mk = _key(os.path.splitext(os.path.basename(movie))[0])
    best, score = None, 0
    for s in subs:
        sk = _key(os.path.splitext(os.path.basename(s))[0])
        n = 0
        while n < min(len(mk), len(sk)) and mk[n] == sk[n]: n += 1
        sc = n + (5 if os.path.dirname(s) == os.path.dirname(movie) else 0)
        if (n >= 5 or (os.path.dirname(s) == os.path.dirname(movie) and n >= 2)) and sc > score: best, score = s, sc
    return best


def ask_gpt(key, system, user, model='gpt-4.1'):
    import urllib.request
    body = json.dumps({'model': model, 'response_format': {'type': 'json_object'},
                       'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]}).encode()
    req = urllib.request.Request('https://api.openai.com/v1/chat/completions', data=body,
                                 headers={'content-type': 'application/json', 'authorization': 'Bearer ' + key})
    try:
        with urllib.request.urlopen(req, timeout=180) as r: j = json.loads(r.read())
    except urllib.error.HTTPError as e:
        msg = e.read().decode('utf-8', 'ignore')[:300]
        if e.code == 401: raise RuntimeError('OpenAI API 키가 맞지 않아요. 다시 실행해서 키를 넣어 주세요.')
        if e.code == 429: raise RuntimeError('OpenAI 사용 한도나 잔액이 부족해요. platform.openai.com에서 확인해 주세요.')
        raise RuntimeError(f'OpenAI 오류 ({e.code}): {msg}')
    return j['choices'][0]['message']['content']


PLAN_SYS = """너는 유튜브 영화 쇼츠 채널 '명화관'의 기획자다. 영화 전체 대사(각 줄 앞 숫자 = 초)와 네가 아는 이 영화의 줄거리를 보고, 쇼츠 1편(완성 1~2분 30초)으로 만들 '재료 구간'을 고르는 추천안 3개를 만든다.

# 명화관 쇼츠의 특징
- 영화 전체가 아니라, 한 가지 이야기(에피소드)만 완결되게 보여 준다. 예: 아빠가 TV 키스캠에 잡혀 집에 못 들어간 이야기 / 하키 선수가 골프장에서 악어와 싸우는 이야기 / 청소부 천재를 찾아낸 교수가 그를 구해 내는 이야기.
- 그 이야기에 필요한 장면을 영화 여러 군데에서 가져온다: ① 인물·상황 소개(이 사람이 왜 특별한지) ② 사건이 터지는 장면 ③ 결과·반전·통쾌한 마무리. 한 장면에서 다 해결되면 구간 1개도 된다.
- 대사가 많고 감정이 터지는 장면, 반전, 웃음, 통쾌함, 감동이 있는 에피소드를 고른다.

# 규칙
- 추천안마다 구간 1~3개. 구간 하나는 40초~3분. 추천안 하나의 구간 합계는 3~6분.
- 구간 경계는 대사가 시작하기 직전·끝난 직후로(대사 중간에서 자르지 않게). 대사 자료의 초 값을 그대로 쓴다.
- 3개 추천안은 서로 다른 에피소드로.
- 제목은 명화관식 두 줄: 윗줄 = 누가·어떤 상황(인물은 이름 말고 역할로), 아랫줄 = "~한 이유 / ~하면 생기는 일 / ~가 한 행동"처럼 궁금증으로 끝. 예: "아빠가 / 집에 못 들어간 이유", "아이스 하키 선수가 / 골프로 전향하면 생기는 일".
- story: 이 쇼츠가 들려줄 이야기 2문장(한국어). why: 왜 조회수가 잘 나올지 한 줄.
- memo: 쇼츠 제작기에 넣을 메모 3줄(줄거리 / 등장인물: 한국어 이름(관계) / 강조할 장면).

JSON만:
{"plans":[{"title1":"윗줄","title2":"아랫줄","story":"...","why":"...","memo":"...","segments":[{"start":초,"end":초,"role":"소개|사건|결과","desc":"이 구간에서 일어나는 일 한 줄"}]}]}"""


def recommend(key, movie, subs, log=print):
    info = probe(movie)
    lines = '\n'.join(f"{int(s['start'])} {s['text'].replace(chr(10), ' ')}" for s in subs)
    if len(lines) > 110000: lines = lines[:110000]
    user = f"영화: {pretty_name(movie)}\n파일 이름: {os.path.basename(movie)}\n영화 길이: {int(info['dur'])}초\n\n대사:\n{lines}"
    log('AI가 영화 전체 대사를 읽고 쇼츠로 만들 이야기를 고르는 중… (30초~1분)')
    plans = json.loads(ask_gpt(key, PLAN_SYS, user)).get('plans', [])
    out = []
    for p in plans[:3]:
        segs = []
        for s in p.get('segments', [])[:3]:
            a, b = float(s.get('start', 0)), float(s.get('end', 0))
            a = max(0, a); b = min(info['dur'], b)
            if b - a < 20: continue
            if b - a > 200: b = a + 200
            segs.append({'start': a, 'end': b, 'role': s.get('role', ''), 'desc': s.get('desc', '')})
        if segs: out.append({**p, 'segments': segs})
    return out


def show_plans(plans, log=print):
    for i, p in enumerate(plans, 1):
        tot = sum(s['end'] - s['start'] for s in p['segments'])
        log(f"\n━━━━━━━━ 추천 {i} ━━━━━━━━")
        log(f"제목:  {p.get('title1','')} / {p.get('title2','')}")
        log(f"이야기: {p.get('story','')}")
        log(f"좋은 이유: {p.get('why','')}")
        for s in p['segments']:
            log(f"   [{s['role']}] {hms(s['start'])} ~ {hms(s['end'])}  {s['desc']}")
        log(f"   재료 길이: 약 {tot/60:.1f}분")


def make_from_plan(movie, subs, segs, out_root, tag='', memo='', log=print):
    """구간 여러 개를 720p로 잘라 순서대로 이어 붙이고, 자막도 이어 붙인 시간에 맞춰요."""
    os.makedirs(out_root, exist_ok=True)
    base = re.sub(r'[\\/:*?"<>|]', '', nfc(os.path.splitext(os.path.basename(movie))[0]))[:40]
    name = f"{base}{tag}_{hms(segs[0]['start']).replace(':','')}-{hms(segs[-1]['end']).replace(':','')}"
    t0 = time.time(); parts = []; clip = []; off = 0.0
    for k, s in enumerate(segs, 1):
        log(f"구간 {k}/{len(segs)}: {hms(s['start'])} ~ {hms(s['end'])} 자르는 중…")
        tmp = f'/tmp/_part{k}.mp4'
        cut_720p(movie, s['start'], s['end'], tmp, log)
        d = probe(tmp)['dur'] or (s['end'] - s['start'])
        for x in subs:
            if x['end'] > s['start'] and x['start'] < s['end']:
                clip.append({'start': max(0, x['start'] - s['start']) + off, 'end': min(x['end'], s['end']) - s['start'] + off, 'text': x['text']})
        parts.append(tmp); off += d
    out_tmp = '/tmp/' + name + '.mp4'
    if len(parts) == 1: shutil.move(parts[0], out_tmp)
    else:
        lst = '/tmp/_list.txt'; open(lst, 'w').write(''.join(f"file '{p}'\n" for p in parts))
        r = subprocess.run(['ffmpeg', '-v', 'error', '-y', '-f', 'concat', '-safe', '0', '-i', lst, '-c', 'copy', '-movflags', '+faststart', out_tmp], capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError('구간을 이어 붙이지 못했어요: ' + r.stderr[-300:])
    open('/tmp/' + name + '.srt', 'w', encoding='utf-8').write(to_srt(clip))
    shutil.copy(out_tmp, os.path.join(out_root, name + '.mp4')); shutil.copy('/tmp/' + name + '.srt', os.path.join(out_root, name + '.srt'))
    if memo: open(os.path.join(out_root, name + '_메모.txt'), 'w', encoding='utf-8').write(memo)
    mb = os.path.getsize(os.path.join(out_root, name + '.mp4')) / 1048576
    log(f"\n완료! ({time.time()-t0:.0f}초) 영상 {off/60:.1f}분 · {mb:.0f}MB · 자막 {len(clip)}줄")
    log(f"구글 드라이브 > {OUT_DIR_NAME} 폴더에 저장했어요:\n  {name}.mp4\n  {name}.srt" + (f"\n  {name}_메모.txt" if memo else ''))
    return name

ROOT='/content/drive/MyDrive'
OUT=ROOT+'/'+OUT_DIR_NAME
print('드라이브에서 영화를 찾는 중…')
VIDS, SUBS = find_files(ROOT)
print('\n🎬 영화 목록  (자막 ✓ = 같은 이름의 자막 파일이 있어요)')
for i,p in enumerate(VIDS,1):
    print(f'  {i:>2}. {pretty_name(p)}   {os.path.getsize(p)/1073741824:.1f}GB   자막 {"✓" if match_sub(p,SUBS) else "-"}')
if not VIDS: print('\n영화 파일을 못 찾았어요. 드라이브에 영화를 올렸는지 확인해 주세요.')


In [ ]:
#@title ② 영화 고르고 AI 추천 받기 { display-mode: "form" }
영화_번호 = 1 #@param {type:"integer"}
import getpass
try:
    MOVIE = VIDS[영화_번호-1]
    print('영화:', pretty_name(MOVIE))
    sp = match_sub(MOVIE, SUBS); INFO = probe(MOVIE)
    SUBS_ALL = load_subs(sp) if sp else embedded_kor_subs(MOVIE, INFO)
    print('자막:', (os.path.basename(sp) if sp else '영화 안에 든 자막'), f'· {len(SUBS_ALL)}줄')
    if len(SUBS_ALL) < 30:
        print('\n한국어 자막을 찾지 못했어요. 영화와 같은 폴더에 smi나 srt 자막을 올리고 ①부터 다시 눌러 주세요.')
    else:
        if not globals().get('OPENAI_KEY'):
            OPENAI_KEY = getpass.getpass('OpenAI API 키를 붙여 넣고 Enter (화면에 안 보여요, 저장되지 않아요): ').strip()
        PLANS = recommend(OPENAI_KEY, MOVIE, SUBS_ALL)
        show_plans(PLANS)
        print('\n마음에 드는 추천 번호를 ③ 칸에 넣고 ▶ 버튼을 누르세요.')
except NameError:
    print('먼저 위의 ① 칸 ▶ 버튼을 눌러 주세요.')
except IndexError:
    print('영화 번호를 확인해 주세요. ① 목록에 있는 번호를 넣어야 해요.')
except Exception as e:
    if '키가 맞지' in str(e): OPENAI_KEY = ''
    print('문제가 생겼어요:', e)


In [ ]:
#@title ③ 고른 추천으로 재료 만들기 { display-mode: "form" }
추천_번호 = 1 #@param {type:"integer"}
#@markdown 추천 말고 직접 구간을 정하려면 추천_번호를 0으로 하고 아래 시간을 넣으세요.
직접_시작 = "" #@param {type:"string"}
직접_끝 = "" #@param {type:"string"}
try:
    if 추천_번호 > 0:
        p = PLANS[추천_번호-1]
        memo = f"제목: {p.get('title1','')} / {p.get('title2','')}\n{p.get('memo','')}"
        make_from_plan(MOVIE, SUBS_ALL, p['segments'], OUT, tag=f'_추천{추천_번호}', memo=memo)
        print('\n📋 쇼츠 제작기 메모 칸에 붙여 넣을 내용:\n' + memo)
    else:
        a, b = parse_time(직접_시작), parse_time(직접_끝)
        if a is None or b is None or b <= a: raise ValueError('직접_시작·직접_끝 시간을 확인해 주세요. 예) 1:36:30 / 1:41:30')
        make_from_plan(MOVIE, SUBS_ALL, [{'start':a,'end':b,'role':'','desc':''}], OUT)
except NameError:
    print('먼저 ①, ② 칸을 차례로 실행해 주세요.')
except IndexError:
    print('추천 번호를 확인해 주세요 (1~3).')
except Exception as e:
    print('문제가 생겼어요:', e)
